<a href="https://colab.research.google.com/github/ronan-cunha/machine-learning-course/blob/main/Notebooks/aula_3_visualizacao.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np

# Base de votos por país#
url_votes = "https://raw.githubusercontent.com/rfordatascience/tidytuesday/master/data/2021/2021-03-23/unvotes.csv"
df_votes = pd.read_csv(url_votes)

# Base com data das votações
url_roll_calls = "https://raw.githubusercontent.com/rfordatascience/tidytuesday/master/data/2021/2021-03-23/roll_calls.csv"
df_roll = pd.read_csv(url_roll_calls)

In [ ]:
print(df_votes.shape)
print(df_votes['rcid'].nunique())
df_votes.head()

In [ ]:
print(df_roll.shape)
print(df_roll['rcid'].nunique())
df_roll.head()

In [ ]:
df_roll['year'] = pd.to_datetime(df_roll['date']).dt.year
df_unga = pd.merge(
    df_votes,
    df_roll,
    on='rcid',
    how='inner',
    validate='many_to_one'  # Boa prática para segurança do código
)

In [ ]:
print(df_unga.shape)
print(df_unga['rcid'].nunique())

In [ ]:
print(df_unga.info())

In [ ]:
df_unga.head()

In [ ]:
df_unga.describe()

In [ ]:
df_country = df_unga[['country_code', 'country']].drop_duplicates()
df_country.sort_values(by='country')

In [ ]:
df_country[df_country['country_code'].isin(['BR', 'US'])]

In [ ]:
df_country.where(df_country['country_code'].isin(['BR', 'US'])).dropna()

In [ ]:
# 2. Operações de String vetorizadas (.str): Filtrando resoluções sobre Desarmamento e Sanções
nuclear_res = df_unga[df_unga['descr'].str.contains('nuclear|disarmament', case=False, na=False)]

In [ ]:
nuclear_res['descr'].iloc[0]

'TO ADOPT AD HOC POLITICAL COMMITTEE DRAFT RESOLUTION (A/2762) DECLARING THAT THE FOREIGN ARMED FORCES STILL IN BURMESE TERRITORY SHOULD SUBMIT TO DISARMAMENT AND INTERNMENT.'

In [ ]:
from wordcloud import WordCloud, STOPWORDS
import matplotlib.pyplot as plt

# 1. Unir todo o texto da coluna 'descr' em uma única string tratada
texto_completo = " ".join(nuclear_res['descr'].dropna().astype(str))

# 2. Configurar palavras que devem ser ignoradas (artigos, preposições, etc.)
stopwords_custom = set(STOPWORDS)

# Adicionar stopwords específicas
#stopwords_custom.update(["assembly","draft"])

# 3. Criar o objeto WordCloud
wordcloud = WordCloud(
    width=1000,
    height=500,
    background_color='white',
    stopwords=stopwords_custom,
    max_words=100,
    min_word_length=3
).generate(texto_completo)

# 4. Plotar o gráfico com Matplotlib
plt.figure(figsize=(12, 6))
plt.imshow(wordcloud, interpolation='bilinear')
plt.axis('off')
plt.title('Palavras mais Frequentes nas Resoluções sobre Desarmamento Nuclear', fontsize=14, pad=15, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# 3. Seleção via .loc e .iloc
nuclear_res = nuclear_res.loc[
    (nuclear_res['year'] >= 1960),
    ['year', 'rcid', 'country', 'country_code', 'vote', 'descr']
]
nuclear_res.head()

In [ ]:
nuclear_res['vote'].value_counts(dropna=False)

In [ ]:
# Recodificar escala de concordância diplomática (Signorino & Ritter, 1999):
# A Favor = 1.0 | Abstenção = 0.5 | Contra = 0.0
vote_map = {'yes': 1.0, 'abstain': 0.5, 'no': 0.0}
nuclear_res['vote_score'] = nuclear_res['vote'].map(vote_map)

In [ ]:
nuclear_res.head()

In [ ]:
nuclear_res1 = nuclear_res.groupby(['year', 'rcid'])['vote_score'].mean()
nuclear_res1 = nuclear_res1.reset_index(drop=False)

In [ ]:
import seaborn as sns

plt.figure(figsize=(8, 4))
sns.set_theme(style="whitegrid")

# 1. Pontos individuais para cada resolução
sns.scatterplot(
    data=nuclear_res1,
    x='year',
    y='vote_score',
    alpha=0.3,
    color='#4c72b0',
    s=30,
    edgecolor=None
)

# 2. Linha de tendência suavizada
sns.regplot(
    data=nuclear_res1,
    x='year',
    y='vote_score',
    scatter=False,
    lowess=True,
    color='#c44e52',
    line_kws={'linewidth': 3, 'label': 'Tendência Suavizada'}
)

plt.title('Dispersão das Resoluções Nucleares na AGNU por Ano', fontsize=14, fontweight='bold', pad=15)
plt.xlabel('Ano', fontsize=12)
plt.ylabel('Média de Voto da Resolução', fontsize=12)
plt.ylim(-0.05, 1.05)
plt.legend(loc='lower left')
plt.tight_layout()
plt.show()

In [ ]:
# Agregar dados por ano: Média do apoio e Contagem de resoluções
df_yearly = nuclear_res1.groupby('year').agg(
    mean_score=('vote_score', 'mean'),
    total_resolutions=('rcid', 'count')
).reset_index()

fig, ax1 = plt.subplots(figsize=(12, 6))
sns.set_theme(style="white")

# Eixo 1: Barras para o número de resoluções por ano
ax1.bar(df_yearly['year'], df_yearly['total_resolutions'], color='#cbd5e1', alpha=0.7, label='Nº de Resoluções')
ax1.set_xlabel('Ano', fontsize=12)
ax1.set_ylabel('Quantidade de Resoluções Votadas', fontsize=12, color='#475569')
ax1.tick_params(axis='y', labelcolor='#475569')
ax1.grid(False)

# Eixo 2: Linha para o nível médio de concordância
ax2 = ax1.twinx()
ax2.plot(df_yearly['year'], df_yearly['mean_score'], color='#2563eb', linewidth=2.5, label='Concordância Média')
ax2.set_ylabel('Média de Concordância Global (0 a 1)', fontsize=12, color='#2563eb')
ax2.tick_params(axis='y', labelcolor='#2563eb')
ax2.set_ylim(-0.05, 1.05)
ax2.grid(True, linestyle='--', alpha=0.5)

plt.title('Volume de Resoluções Nucleares vs. Nível de Apoio Global na AGNU', fontsize=14, fontweight='bold', pad=15)
fig.tight_layout()
plt.show()

In [ ]:
# Agregar dados por ano: Média do apoio e Contagem de resoluções
df_yearly = nuclear_res1.groupby('year').agg(
    mean_score=('vote_score', 'mean'),
    sd_score=('vote_score', 'std'),
    total_resolutions=('rcid', 'count')
).reset_index()

fig, ax1 = plt.subplots(figsize=(12, 6))
sns.set_theme(style="white")

# Eixo 1: Barras para o número de resoluções por ano
ax1.bar(df_yearly['year'], df_yearly['total_resolutions'], color='#cbd5e1', alpha=0.7, label='Nº de Resoluções')
ax1.set_xlabel('Ano', fontsize=12)
ax1.set_ylabel('Quantidade de Resoluções Votadas', fontsize=12, color='#475569')
ax1.tick_params(axis='y', labelcolor='#475569')
ax1.grid(False)

# Eixo 2: Linha para o nível médio de concordância
ax2 = ax1.twinx()
ax2.plot(df_yearly['year'], df_yearly['mean_score'], color='#2563eb', linewidth=2.5, label='Concordância Média')
ax2.plot(df_yearly['year'], df_yearly['sd_score'], color='red', linewidth=2.5, label='Desv. pad. Concordância ')
ax2.tick_params(axis='y', labelcolor='#2563eb')
ax2.set_ylim(-0.05, 1.05)
ax2.grid(True, linestyle='--', alpha=0.5)

plt.title('Volume de Resoluções Nucleares vs. Nível de Apoio Global na AGNU', fontsize=14, fontweight='bold', pad=15)
fig.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(14, 5))
sns.boxplot(
    data=nuclear_res1,
    x="year",
    y="vote_score",
    color="#cbd5e1",  # Cinza neutro suave
    showfliers=False,  # Oculta outliers do boxplot pois o stripplot plota todos os pontos
    boxprops=dict(linewidth=1.2, edgecolor="#475569"),
    whiskerprops=dict(color="#475569"),
    capprops=dict(color="#475569"),
    medianprops=dict(color="#2563eb", linewidth=2),  # Mediana em azul de destaque
)
plt.xticks(rotation=90, fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
nuclear_res.head()

In [ ]:
# 3. Seleção via .loc e .iloc
nuclear_res2 = nuclear_res.loc[
    (nuclear_res['year'] >= 1991) & (nuclear_res['country_code'].isin(['BR', 'US', ])),
    ['year', 'rcid', 'country', 'country_code', 'vote_score', 'descr']
]
nuclear_res2

In [ ]:
piv_recent = nuclear_res2[nuclear_res2['year'] >= 1990].pivot_table(
    index='rcid',
    columns='country_code',
    values='vote_score'
)

In [ ]:
piv_recent.head()

In [ ]:
# 3. Índice de Concordância Diplomática com o Brasil (BRA):
# Fórmula: 1 - |Voto_Brasil - Voto_OutroPaís|
align_bra = (1 - (piv_recent.sub(piv_recent['BR'], axis=0)).abs()).mean()
align_bra.sort_values(ascending=False)
align_bra

# A Autonomia e as Grandes Potências:

1) Calcule o Índice de Concordância Diplomática do Brasil (BRA) com os todos os países e agrege ao longo do tempo.

2) Calcule o Índice de Concordância Diplomática do Brasil (BRA) com os EUA (USA) e com a China (CHN) por década.